# Análisis AP desde cero
Este notebook reconstruye el análisis del archivo `ap(1).xlsx` sin usar pandas/openpyxl.

In [ ]:
import os, json, math, statistics, zipfile, xml.etree.ElementTree as ET, re
from collections import defaultdict, Counter
from datetime import datetime, timedelta, date
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

input_path = "/mnt/data/ap(1).xlsx"
outdir = "/mnt/data/analisis_ap_output"
os.makedirs(outdir, exist_ok=True)

ns = {'a':'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
      'r':'http://schemas.openxmlformats.org/officeDocument/2006/relationships'}

def col_to_idx(col):
    idx = 0
    for ch in col:
        if ch.isalpha():
            idx = idx * 26 + (ord(ch.upper()) - 64)
    return idx - 1

def parse_date_any(x):
    if isinstance(x, datetime):
        if x.year <= 1900:
            return None
        return x
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None
        for fmt in ("%d-%m-%Y", "%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d"):
            try:
                return datetime.strptime(s, fmt)
            except:
                pass
    return None

def parse_xlsx_records(path):
    zf = zipfile.ZipFile(path)
    wb = ET.fromstring(zf.read("xl/workbook.xml"))
    sheets = [(s.attrib["name"], s.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"])
              for s in wb.find("a:sheets", ns)]
    rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
    relmap = {r.attrib["Id"]: r.attrib["Target"] for r in rels}

    styles = ET.fromstring(zf.read("xl/styles.xml"))
    custom_numfmts = {}
    numfmts = styles.find("a:numFmts", ns)
    if numfmts is not None:
        for fmt in numfmts:
            custom_numfmts[int(fmt.attrib["numFmtId"])] = fmt.attrib["formatCode"]
    cellxfs = styles.find("a:cellXfs", ns)
    style_numfmt = []
    if cellxfs is not None:
        for xf in cellxfs:
            style_numfmt.append(int(xf.attrib.get("numFmtId", 0)))
    date_builtin = {14,15,16,17,18,19,20,21,22,45,46,47}

    def is_date_style(style_idx):
        if style_idx is None or style_idx >= len(style_numfmt):
            return False
        numfmt = style_numfmt[style_idx]
        if numfmt in date_builtin:
            return True
        fmt = custom_numfmts.get(numfmt, "").lower()
        return any(tok in fmt for tok in ["yy","dd","mm","h","ss"]) and not fmt.startswith("0")

    def parse_sheet(xml_bytes):
        root = ET.fromstring(xml_bytes)
        rows = []
        for row in root.findall(".//a:sheetData/a:row", ns):
            vals = {}
            for c in row.findall("a:c", ns):
                ref = c.attrib.get("r", "")
                m = re.match(r"([A-Z]+)", ref)
                if not m:
                    continue
                ci = col_to_idx(m.group(1))
                t = c.attrib.get("t")
                s = int(c.attrib.get("s", 0)) if "s" in c.attrib else None
                v = c.find("a:v", ns)
                isnode = c.find("a:is", ns)
                val = None
                if t == "inlineStr" and isnode is not None:
                    texts = [x.text or "" for x in isnode.findall(".//a:t", ns)]
                    val = "".join(texts)
                elif v is not None:
                    txt = v.text
                    if txt is None:
                        val = None
                    else:
                        if is_date_style(s):
                            try:
                                val = datetime(1899, 12, 30) + timedelta(days=float(txt))
                            except:
                                val = txt
                        else:
                            try:
                                if "." in txt or "e" in txt.lower():
                                    num = float(txt)
                                    val = int(num) if num.is_integer() else num
                                else:
                                    val = int(txt)
                            except:
                                val = txt
                vals[ci] = val
            if vals:
                maxci = max(vals)
                rows.append([vals.get(i) for i in range(maxci+1)])
        return rows

    out = {}
    for name, rid in sheets:
        rows = parse_sheet(zf.read(relmap[rid].lstrip("/")))
        if not rows:
            out[name] = []
            continue
        header = rows[0]
        recs = []
        for row in rows[1:]:
            row = row + ([None] * (len(header) - len(row)))
            recs.append({header[i]: row[i] for i in range(len(header))})
        out[name] = recs
    return out

book = parse_xlsx_records(input_path)
ops_tn = book["Operaciones TN"]
vig_tn = book["Vigentes e Incumplidos TN"]

ops_lookup = {r["nm_operacion"]: r for r in ops_tn}
ops = {}
for rec in vig_tn:
    op = rec["nm_operacion"]
    s = ops.setdefault(op, {
        "nm_operacion": op,
        "fecha_operacion": parse_date_any(rec["fecha_operacion"]),
        "fecha_confirmacion_final": None,
        "fecha_autorizada": parse_date_any(rec["Fecha_autorizada"]),
        "monto_usd": rec["monto"],
        "valuacion_mxn": rec["Valuación"],
        "req_capital_max": rec["Requerimiento_capital"] or 0,
        "plazo": rec["Plazo"],
        "dias_adeudo_reales_max": rec["Dias_adeudo_reales"],
        "nm_portafolio": (rec["nm_portafolio"] or "").strip() if isinstance(rec["nm_portafolio"], str) else rec["nm_portafolio"],
    })
    fc = parse_date_any(rec["fecha_confirmacion"])
    if fc and (s["fecha_confirmacion_final"] is None or fc > s["fecha_confirmacion_final"]):
        s["fecha_confirmacion_final"] = fc
    if rec["Requerimiento_capital"] and rec["Requerimiento_capital"] > s["req_capital_max"]:
        s["req_capital_max"] = rec["Requerimiento_capital"]
    val = rec["Dias_adeudo_reales"]
    if val is not None and (s["dias_adeudo_reales_max"] is None or val > s["dias_adeudo_reales_max"]):
        s["dias_adeudo_reales_max"] = val

for op, s in ops.items():
    r = ops_lookup.get(op, {})
    s["justificacion"] = r.get("justificacion")
    s["clasificacion"] = r.get("clasificacion_prepago")
    s["hora_rec"] = r.get("hora_conf_recibimos")

ops = sorted(ops.values(), key=lambda x: x["fecha_operacion"])

yearly = defaultdict(lambda: {"count": 0, "usd": 0.0})
monthly = defaultdict(lambda: {"count": 0, "usd": 0.0})
weekday = Counter()
plazo_weekday = defaultdict(Counter)
doc_metrics = defaultdict(lambda: {"ops": 0, "missing_just": 0, "missing_class": 0, "missing_rec_time": 0})

for s in ops:
    yr = s["fecha_operacion"].year
    ym = s["fecha_operacion"].strftime("%Y-%m")
    yearly[yr]["count"] += 1
    yearly[yr]["usd"] += s["monto_usd"]
    monthly[ym]["count"] += 1
    monthly[ym]["usd"] += s["monto_usd"]
    wd = s["fecha_operacion"].strftime("%A")
    weekday[wd] += 1
    plazo_weekday[s["plazo"]][wd] += 1
    doc_metrics[yr]["ops"] += 1
    if not s["justificacion"]:
        doc_metrics[yr]["missing_just"] += 1
    if not s["clasificacion"]:
        doc_metrics[yr]["missing_class"] += 1
    if s["hora_rec"] in (None, "00:00:00"):
        doc_metrics[yr]["missing_rec_time"] += 1

def daterange(start, end):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

exp_actual = defaultdict(float)
for s in ops:
    start = s["fecha_operacion"].date()
    end = (s["fecha_confirmacion_final"] or s["fecha_autorizada"] or s["fecha_operacion"]).date()
    for d in daterange(start, end):
        exp_actual[d] += s["valuacion_mxn"] or 0

# Produce same charts + summary as needed...
print("Resumen:")
print("Operaciones TN:", len(ops))
print("Volumen USD total:", round(sum(s["monto_usd"] for s in ops), 2))
print("Pico de exposición abierta MXN:", round(max(exp_actual.values()), 2))
print("Día pico:", max(exp_actual, key=lambda d: exp_actual[d]))